**Bronze, loaded as-is**

In [1]:

from pyspark.sql import functions as F

bronze = (
    spark.read.option("header", True).option("inferSchema", False)
    .csv("Files/raw/Online Retail.csv")
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_ingested_at", F.current_timestamp())
)
bronze.write.mode("overwrite").format("delta").saveAsTable("bronze_online_retail")
print(bronze.count())  # save this number for your data-quality page

StatementMeta(, 19587fa8-40a7-407a-9deb-d06ca85445ec, 3, Finished, Available, Finished, False)

541909


In [6]:
%%sql
select count(*) from bronze_online_retail

StatementMeta(, 19587fa8-40a7-407a-9deb-d06ca85445ec, 12, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

**Silver, typed with flags instead of deletions**

In [5]:
%%sql

CREATE OR REPLACE TABLE silver_sales AS
WITH typed AS (
  SELECT DISTINCT
    InvoiceNo                                        AS invoice_no,
    StockCode                                        AS stock_code,
    NULLIF(TRIM(Description), '')                    AS description,
    CAST(Quantity AS INT)                            AS quantity,
    to_timestamp(InvoiceDate, 'M/d/yyyy H:mm')       AS invoice_ts,
    CAST(UnitPrice AS DECIMAL(10,2))                 AS unit_price,
    CAST(CAST(CustomerID AS DOUBLE) AS INT)          AS customer_id,
    TRIM(Country)                                    AS country
  FROM bronze_online_retail
)
SELECT
  *,
  CAST(quantity * unit_price AS DECIMAL(12,2))       AS line_amount,
  invoice_no LIKE 'C%'                               AS is_cancellation,
  stock_code RLIKE '^[0-9]{5}'                       AS is_product,
  customer_id IS NOT NULL                            AS has_customer,
  ( invoice_no NOT LIKE 'C%'
    AND quantity > 0
    AND unit_price > 0
    AND stock_code RLIKE '^[0-9]{5}' )               AS is_valid_sale
FROM typed;

StatementMeta(, 19587fa8-40a7-407a-9deb-d06ca85445ec, 11, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [7]:
%%sql
select * from silver_sales

StatementMeta(, 19587fa8-40a7-407a-9deb-d06ca85445ec, 13, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 13 fields>

**cleaning-rules table**

In [1]:
%%sql
SELECT 'Raw rows (Bronze)'            AS rule, COUNT(*) AS rows_affected FROM bronze_online_retail
UNION ALL SELECT 'After exact-duplicate removal', COUNT(*) FROM silver_sales
UNION ALL SELECT 'Cancellations (kept, flagged)', COUNT(*) FROM silver_sales WHERE is_cancellation
UNION ALL SELECT 'Missing customer ID (kept, flagged)', COUNT(*) FROM silver_sales WHERE NOT has_customer
UNION ALL SELECT 'Non-product lines (kept, flagged)', COUNT(*) FROM silver_sales WHERE NOT is_product
UNION ALL SELECT 'Zero or negative price', COUNT(*) FROM silver_sales WHERE unit_price <= 0
UNION ALL SELECT 'Valid sale lines', COUNT(*) FROM silver_sales WHERE is_valid_sale;

StatementMeta(, 2e77c6d4-82b6-4674-ac3f-7d1076f2cde0, 2, Finished, Available, Finished, False)

<Spark SQL result set with 7 rows and 2 fields>

In [2]:
%%sql
SELECT
  COUNT(*) FILTER (WHERE invoice_ts IS NULL) AS null_dates,
  MIN(invoice_ts) AS first_invoice,
  MAX(invoice_ts) AS last_invoice
FROM silver_sales;

StatementMeta(, 2e77c6d4-82b6-4674-ac3f-7d1076f2cde0, 3, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

In [3]:
%%sql
SELECT
  ROUND(SUM(CASE WHEN NOT has_customer THEN line_amount END)
        / SUM(line_amount) * 100, 1) AS pct_revenue_no_customer
FROM silver_sales
WHERE is_valid_sale;

StatementMeta(, 2e77c6d4-82b6-4674-ac3f-7d1076f2cde0, 4, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

**date dimension**

In [1]:
%%sql
CREATE OR REPLACE TABLE gold_dim_date AS
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT)  AS date_key,
  d                                        AS date,
  year(d)                                  AS year,
  month(d)                                 AS month_num,
  date_format(d, 'MMM yyyy')               AS month_label,
  CAST(date_trunc('month', d) AS DATE)     AS month_start,
  date_format(d, 'E')                      AS weekday
FROM (
  SELECT explode(sequence(DATE'2010-12-01', DATE'2011-12-31', INTERVAL 1 DAY)) AS d
);

StatementMeta(, 83d9f6ad-4b9b-469d-9c7a-18d723a1103e, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
%%sql
Select * from gold_dim_date

StatementMeta(, 83d9f6ad-4b9b-469d-9c7a-18d723a1103e, 3, Finished, Available, Finished, False)

<Spark SQL result set with 396 rows and 7 fields>

**product dimension**

In [7]:
%%sql
CREATE OR REPLACE TABLE gold_dim_product AS
WITH codes AS (
  SELECT DISTINCT stock_code FROM silver_sales WHERE is_product
),
best AS (
  SELECT stock_code, MAX_BY(description, cnt) AS description
  FROM (
    SELECT stock_code, description, COUNT(*) AS cnt
    FROM silver_sales
    WHERE is_product AND description IS NOT NULL
    GROUP BY stock_code, description
  )
  GROUP BY stock_code
)
SELECT c.stock_code,
       COALESCE(b.description, 'Unknown product') AS description
FROM codes c
LEFT JOIN best b USING (stock_code);

StatementMeta(, 83d9f6ad-4b9b-469d-9c7a-18d723a1103e, 9, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [9]:
%%sql
select * from gold_dim_product

StatementMeta(, 83d9f6ad-4b9b-469d-9c7a-18d723a1103e, 11, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 2 fields>

In [11]:
%%sql
CREATE OR REPLACE TABLE gold_dim_customer AS
WITH cust AS (
  SELECT
    customer_id,
    MAX_BY(country, invoice_ts)             AS country,
    CAST(MIN(invoice_ts) AS DATE)           AS first_purchase_date,
    CAST(MAX(invoice_ts) AS DATE)           AS last_purchase_date,
    COUNT(DISTINCT invoice_no)              AS order_count
  FROM silver_sales
  WHERE is_valid_sale AND has_customer
  GROUP BY customer_id
)
SELECT
  customer_id, country, first_purchase_date, last_purchase_date, order_count,
  CASE
    WHEN first_purchase_date >= DATE'2011-09-10' THEN 'Too new'
    WHEN last_purchase_date  <  DATE'2011-09-10' THEN 'Churned'
    ELSE 'Active'
  END AS customer_status
FROM cust
UNION ALL
SELECT -1, 'Unknown', CAST(NULL AS DATE), CAST(NULL AS DATE),
       CAST(0 AS BIGINT), 'Unknown';

StatementMeta(, 83d9f6ad-4b9b-469d-9c7a-18d723a1103e, 13, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [12]:
%%sql
CREATE OR REPLACE TABLE gold_fact_sales AS
SELECT
  invoice_no,
  COALESCE(customer_id, -1)                    AS customer_id,
  stock_code,
  CAST(date_format(invoice_ts, 'yyyyMMdd') AS INT) AS date_key,
  invoice_ts, country, quantity, unit_price, line_amount
FROM silver_sales
WHERE is_valid_sale;

CREATE OR REPLACE TABLE gold_fact_returns AS
SELECT
  invoice_no,
  COALESCE(customer_id, -1)                    AS customer_id,
  stock_code,
  CAST(date_format(invoice_ts, 'yyyyMMdd') AS INT) AS date_key,
  invoice_ts, country, quantity, unit_price, line_amount
FROM silver_sales
WHERE is_cancellation;

StatementMeta(, 83d9f6ad-4b9b-469d-9c7a-18d723a1103e, 15, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [13]:
%%sql
SELECT 'fact_sales rows' AS check_name, COUNT(*) AS value FROM gold_fact_sales
UNION ALL SELECT 'fact_returns rows', COUNT(*) FROM gold_fact_returns
UNION ALL SELECT 'customers (excl. Unknown)', COUNT(*) FROM gold_dim_customer WHERE customer_id <> -1
UNION ALL SELECT 'Active', COUNT(*) FROM gold_dim_customer WHERE customer_status = 'Active'
UNION ALL SELECT 'Churned', COUNT(*) FROM gold_dim_customer WHERE customer_status = 'Churned'
UNION ALL SELECT 'Too new', COUNT(*) FROM gold_dim_customer WHERE customer_status = 'Too new'
UNION ALL SELECT 'Orphan sales (no customer match)', COUNT(*)
  FROM gold_fact_sales f LEFT JOIN gold_dim_customer c USING (customer_id)
  WHERE c.customer_id IS NULL;

StatementMeta(, 83d9f6ad-4b9b-469d-9c7a-18d723a1103e, 16, Finished, Available, Finished, False)

<Spark SQL result set with 7 rows and 2 fields>

In [1]:
%%sql
CREATE OR REPLACE TABLE gold_cohort_retention AS
WITH first_buy AS (
  SELECT customer_id,
         CAST(date_trunc('month', MIN(invoice_ts)) AS DATE) AS cohort_month
  FROM silver_sales
  WHERE is_valid_sale AND has_customer
  GROUP BY customer_id
),
activity AS (
  SELECT DISTINCT customer_id,
         CAST(date_trunc('month', invoice_ts) AS DATE) AS activity_month
  FROM silver_sales
  WHERE is_valid_sale AND has_customer
    AND invoice_ts < TIMESTAMP'2011-12-01 00:00:00'
),
joined AS (
  SELECT f.cohort_month, a.customer_id,
         CAST(months_between(a.activity_month, f.cohort_month) AS INT) AS month_index
  FROM first_buy f
  JOIN activity a USING (customer_id)
),
sizes AS (
  SELECT cohort_month, COUNT(*) AS cohort_size
  FROM first_buy
  GROUP BY cohort_month
)
SELECT j.cohort_month,
       j.month_index,
       COUNT(DISTINCT j.customer_id)                         AS active_customers,
       s.cohort_size,
       ROUND(COUNT(DISTINCT j.customer_id) / s.cohort_size, 4) AS retention_rate
FROM joined j
JOIN sizes s USING (cohort_month)
WHERE j.cohort_month < DATE'2011-12-01'
GROUP BY j.cohort_month, j.month_index, s.cohort_size;

StatementMeta(, bfdd62e2-6de0-4c5e-9df8-b6b1ef762bcc, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [4]:
%%sql
select * FROM gold_cohort_retention order by cohort_month,month_index asc;

StatementMeta(, 8ce52593-39f8-4a28-b954-8fddd9e512d6, 5, Finished, Available, Finished, False)

<Spark SQL result set with 78 rows and 5 fields>

In [1]:
%%sql
CREATE OR REPLACE TABLE dq_cleaning_rules AS
SELECT 1 AS sort_order, 'Raw rows (Bronze)' AS rule, COUNT(*) AS rows_affected FROM bronze_online_retail
UNION ALL SELECT 2, 'After exact-duplicate removal', COUNT(*) FROM silver_sales
UNION ALL SELECT 3, 'Cancellations (kept, flagged)', COUNT(*) FROM silver_sales WHERE is_cancellation
UNION ALL SELECT 4, 'Missing customer ID (kept, flagged)', COUNT(*) FROM silver_sales WHERE NOT has_customer
UNION ALL SELECT 5, 'Non-product lines (kept, flagged)', COUNT(*) FROM silver_sales WHERE NOT is_product
UNION ALL SELECT 6, 'Zero or negative price', COUNT(*) FROM silver_sales WHERE unit_price <= 0
UNION ALL SELECT 7, 'Valid sale lines', COUNT(*) FROM silver_sales WHERE is_valid_sale;

StatementMeta(, e72c362e-a084-47de-8656-abe56e39430f, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
%%sql
SELECT * FROM dq_cleaning_rules

StatementMeta(, e72c362e-a084-47de-8656-abe56e39430f, 3, Finished, Available, Finished, False)

<Spark SQL result set with 7 rows and 3 fields>

In [2]:
%%sql
SELECT COUNT(*) AS dec_2011_rows
FROM gold_cohort_retention
WHERE cohort_month = DATE'2011-12-01'

StatementMeta(, bfdd62e2-6de0-4c5e-9df8-b6b1ef762bcc, 3, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [1]:
%%sql
-- Revenue concentration: share of revenue from the top 10% of customers
WITH c AS (
  SELECT customer_id, SUM(line_amount) AS rev,
         NTILE(10) OVER (ORDER BY SUM(line_amount) DESC) AS decile
  FROM gold_fact_sales
  WHERE customer_id <> -1 and invoice_ts < TIMESTAMP'2011-12-01 00:00:00'
  GROUP BY customer_id
)
SELECT ROUND(SUM(CASE WHEN decile = 1 THEN rev END) / SUM(rev) * 100, 1) AS top10_pct_revenue
FROM c;

StatementMeta(, c19d8a4f-d023-4641-bff2-99767ee5e46e, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [2]:
%%sql
SELECT ROUND(
  SUM(CASE WHEN c.customer_status = 'Churned' THEN f.line_amount END)
  / SUM(f.line_amount) * 100, 1) AS churned_revenue_pct
FROM gold_fact_sales f
JOIN gold_dim_customer c USING (customer_id)
WHERE f.customer_id <> -1 and invoice_ts < TIMESTAMP'2011-12-01 00:00:00'

StatementMeta(, c19d8a4f-d023-4641-bff2-99767ee5e46e, 3, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>